# 05 LLM 판단 노트북

광고 1건을 넣으면 **검사기준표 30개 항목**을 판정하고, 항목마다 근거 조항과 원문 인용을 붙여 줍니다.

```
광고 JSON(주연 님 전처리)
  → ① 상품유형 정하기
  → ② 코드 판정 (숫자 기준: 대비, 글자 크기, 노출 시간, 심사필 …)
  → ③ 근거 카드 검색 (04 RAG: BM25 + BGE-M3 + 리랭커)
  → ④ LLM 판단 (문구 항목 판정 + 모든 항목에 근거 인용)
  → ⑤ 근거 검증 (인용한 카드·문장이 진짜 있는지 코드로 확인)
  → ⑥ 결과표 + 종합 의견 저장
```

**순서**
1. 준비 (API 키, 경로)
2. 검사기준표 불러오기
3. 상품유형 자동 채우기
4. 광고 불러오기 + 문구 정리
5. 코드 판정
6. 근거 카드 검색
7. 프롬프트 만들기
8. LLM 호출 (OpenAI / Claude)
9. 근거 검증
10. 광고 1건 판정
11. 여러 광고 · 여러 모델 비교

> 판정 함수는 `rag/judge.py`에 있고, 이 노트북은 그 모듈을 불러 단계별로 보여 줍니다. 로직을 고칠 때는 `judge.py`를 고치세요.
> 광고 여러 건은 명령줄 실행기 `python run_judge.py --all-images --skip-existing`로 돌립니다 (README 참고).

## 1. 준비
처음 한 번만 터미널에서 설치합니다 (가상환경을 켠 상태에서).
```bash
pip install openai anthropic python-dotenv
```
API 키는 `rag/.env` 파일에 넣습니다. **키를 노트북 안에 직접 쓰지 마세요.**
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...      # Claude는 나중에 추가해도 됩니다
```

In [ ]:
import os
import inspect
import pandas as pd

# 판정 함수와 설정은 rag/judge.py에 있습니다 (run_judge.py와 같은 코드).
# rag_search(04 검색 장치)도 함께 불러오므로 30초~1분 걸립니다.
import judge
from judge import *

print('광고 JSON 위치:', REPO_DIR / 'data/processed')
print('OpenAI 키:', '있음' if os.getenv('OPENAI_API_KEY') else '없음')
print('Claude 키:', '있음' if os.getenv('ANTHROPIC_API_KEY') else '없음')
print('기본 모델:', MODELS)   # 바꾸려면 MODELS['openai'] = 'gpt-6.2-sol' 처럼 (judge.MODELS와 같은 dict)
pd.set_option('display.max_colwidth', 120)   # 표에서 긴 글이 너무 잘리지 않게

04 노트북에서 만든 검색 장치를 불러옵니다 (`rag_search.py`). 모델을 불러오느라 30초~1분 걸립니다.

In [ ]:
search_ids('영상 광고에서 위험고지가 짧게만 노출됨', '투자성', k=3)

## 2. 검사기준표 불러오기
`data/checks.csv`는 Google 시트의 **검사기준표** 탭에 노트북이 쓰는 열 몇 개를 더한 것입니다.
- `products`: 적용 상품유형 (`전체`, `투자성`, `!대출성` = 대출성 제외)
- `media`: 적용 매체 (`all`, `image`, `video`, `banner` …)
- `mode`: `code`(숫자로 판정), `llm`(LLM이 판정), `code+llm`, `skip`(VLM 단계에서 판단)
- `group`: LLM에 함께 묻는 묶음 (AD: 표시 방법·심의 표시, B: 필수 내용, CE: 금지 표현·추천보증)

기준값(TH)은 검사기준표의 임시값입니다. 보정하면 여기만 고치면 됩니다.

In [ ]:
# checks(검사기준표), TH(기준값), VERDICTS는 judge.py에 있습니다.
# 기준값을 바꿔 보려면 TH['contrast'] = 3.5 처럼 고칩니다 (judge.TH와 같은 dict라 판정에 바로 반영).
print(TH)
checks[['check_id', 'item', 'products', 'media', 'mode', 'group']]

## 3. 상품유형 자동 채우기
주연 님 JSON에는 상품유형이 비어 있는 경우가 많아서, **브랜드 · 제목 · 광고 문구의 키워드**로 정합니다.
- 결과가 `전체`면 상품을 특정하지 못한 것(기업 PR 광고 등)이라 공통 항목만 검사합니다.
- `범위밖`은 신용카드처럼 우리 규정 카드로 다루기 어려운 광고입니다.
- 틀린 게 있으면 `PRODUCT_OVERRIDE`에 직접 적으면 그 값이 우선합니다.

In [ ]:
# 틀린 상품유형은 judge.py의 PRODUCT_OVERRIDE에 적거나, 여기서 PRODUCT_OVERRIDE['V112'] = '투자성' 처럼 추가합니다.
prod = pd.DataFrame([{'ad_id': i, **dict(zip(['product', 'by'], infer_product(load_ad(i)))),
                      'brand': load_ad(i).get('brand')} for i in ALL_IDS])
print(prod['product'].value_counts().to_string())
prod[prod['ad_id'].str.startswith('V')].head(30)

## 4. 광고 정리
LLM에 넘길 광고 정보를 한 덩어리로 만듭니다. 위험고지 문단과 그 밖의 문구(헤드라인·수익 표현 등)를 나눠서 넘깁니다.

In [ ]:
ad = load_ad('AD01'); ptype, _ = infer_product(ad)
json.dumps(summarize_ad(ad, ptype), ensure_ascii=False)[:600]

## 5. 코드 판정
숫자로 판정할 수 있는 항목은 LLM에 맡기지 않고 코드로 정합니다.
필수 문구(B1 설명서 권유, B3 투자 위험, B5 설명 받을 권리, B6 예금자보호)는 키워드가 있으면 코드로 `적합`, 없으면 LLM이 판정합니다. LLM은 이 결과를 **바꿀 수 없고**, 근거 조항만 붙입니다.

In [ ]:
code_checks(ad, ptype)

## 6. 근거 카드 검색
LLM에 묻는 묶음(AD / B / CE)마다, 그 묶음의 항목별 질문으로 카드를 8장씩 찾고 합칩니다. 검사기준표의 **근거 카드**도 항상 넣어 줍니다(검색이 놓쳐도 빠지지 않게). 묶음당 최대 16장입니다.

In [ ]:
rows = checks[checks['group'] == 'B']
retrieve(rows[rows.apply(lambda r: applicable(r, ptype, media_of(ad)), axis=1)], ptype)

## 7. 프롬프트
LLM에게 주는 규칙의 핵심:
1. **주어진 카드만** 근거로 쓰고, 인용은 카드 본문에 **그대로 있는 문장**만 씁니다.
2. 코드 판정이 있는 항목은 판정을 바꾸지 않고 근거만 붙입니다.
3. 우리 서비스 기준(숫자가 없는 법 기준을 해석한 것)에 걸린 경우는 `위반`이 아니라 `확인필요`입니다.
4. 위반은 법 기준이 명확하고 광고 문구로 확실할 때만. 애매하면 확인필요입니다.
5. 판정 근거가 된 **광고 문구(ad_quote)**도 그대로 옮기게 해서, 코드로 실제 광고에 있는지 확인합니다.
6. 정해진 JSON 형식으로만 답합니다.

In [ ]:
print(SYSTEM[:300])
print(build_prompt(summarize_ad(ad, ptype), rows.head(2), code_checks(ad, ptype), ['K-B9-1'])[-800:])

## 8. LLM 호출
OpenAI와 Claude를 **같은 함수 모양**으로 부릅니다. 그래야 같은 프롬프트로 두 모델을 비교할 수 있습니다.
키가 없는 쪽은 건너뜁니다.

In [ ]:
print(inspect.getsource(call_llm))

## 9. 근거 검증
LLM 답을 그대로 믿지 않고 코드로 확인합니다. 문제가 있으면 `issues` 열에 적힙니다.
- 판정 값이 정해진 4가지 중 하나인가
- 코드 판정을 바꾸지 않았는가
- 인용한 카드가 이번에 넘겨준 카드인가
- 인용 문장이 그 카드 본문에 **실제로 있는가** (띄어쓰기 무시하고 비교. `…`·`...`·줄바꿈으로 나눈 5자 이상 조각이 모두 인용 카드 원문에 있어야 통과)
- LLM이 옮긴 **광고 문구(ad_quote)가 실제 광고에 있는가** (` / `로 이어 붙였으면 조각마다 확인)
- 카드 원문과 인용 모두 `<개정 …>`, `<신설 …>` 같은 꺾쇠 주석은 지우고 비교

In [ ]:
print(inspect.getsource(validate))

## 10. 광고 1건 판정
`judge_ad('AD01', 'openai')`처럼 부릅니다. 결과는 표로 보여주고 `outputs/{광고ID}__{모델}.json`과 `.csv`에 저장합니다. CSV는 엑셀이나 Google 시트로 열면 잘리지 않고 다 보입니다.

LLM 호출은 묶음당 1번, 광고 1건에 최대 3번입니다. LLM이 `위반`을 낸 묶음은 같은 프롬프트로 한 번 더 물어(자기 일관성 재확인), 2차에도 `위반`일 때만 유지하고 아니면 `확인필요`로 낮춥니다(issues에 `재확인 불일치` 기록, 토큰·재호출 횟수는 결과 JSON에 합산).
C5(혜택·불이익 균형)는 다른 항목 판정이 끝난 뒤 코드로 정합니다. A2~A7·C3 중 확인필요·위반이 하나라도 있으면 `확인필요`, 없으면 `적합`입니다. LLM의 이유는 종합 의견으로 남깁니다.

In [ ]:
judge_ad('AD01', 'openai')

위반·확인필요 항목만 보기:

In [ ]:
r = judge_ad('V101', 'openai')
r[r['verdict'].isin(['위반', '확인필요'])]

## 11. 여러 광고 · 여러 모델 비교
최종 비교 때 씁니다. 같은 광고 목록을 모델마다 돌리고, **항목별 판정이 얼마나 일치하는지**와 **근거 검증 통과율**을 봅니다.
정답표(위반 예시)가 생기면 여기에 정확도 계산을 붙입니다.

In [ ]:
# run_many(ad_ids, providers)는 judge.py에 있습니다. 많은 광고는 명령줄 실행기가 편합니다:
#   python run_judge.py --all-images --skip-existing --workers 4
# 예시: 실제 ETF 광고 5건
# many = run_many(['AD01', 'AD02', 'V101', 'V102', 'V113'])
# many.pivot_table(index=['ad_id', 'check_id'], columns='provider', values='verdict', aggfunc='first')
# many.groupby('provider')['issues'].apply(lambda s: (s == '').mean())   # 근거 검증 통과율